In [2]:
import pandas as pd

DATA_PATH='./data'

In [ ]:
# load from discharge summary notes if they already exist
try:
    notes_df = pd.read_csv(f"{DATA_PATH}/discharge_summary_notes.csv")
    print("Loaded existing discharge summary notes.")
except FileNotFoundError:
    # Load the noteevents data
    chunksize = 10000  # Adjust based on memory
    filtered_data = []

    # load the notes data in chunks
    for chunk in pd.read_csv(f"{DATA_PATH}/NOTEEVENTS.csv.gz", usecols=['CATEGORY', 'TEXT', 'HADM_ID'], chunksize=chunksize):
        filtered_chunk = chunk[chunk["CATEGORY"] == "Discharge summary"]
        filtered_data.append(filtered_chunk)
    notes_df = pd.concat(filtered_data, ignore_index=True)
    notes_df.to_csv(f"{DATA_PATH}/discharge_summary_notes.csv", index=False)

Loaded existing discharge summary notes.


In [15]:
# load the procedures data
procedures_df = pd.read_csv(f"{DATA_PATH}/PROCEDURES_ICD.csv.gz", usecols=['HADM_ID', 'ICD9_CODE'])
# Merge the notes and procedures data on HADM_ID
merged_df = pd.merge(procedures_df, notes_df, on='HADM_ID', how='inner')

# get all procedure codes associated with the admission
procedure_codes_with_adm = merged_df.groupby('HADM_ID')['ICD9_CODE'].apply(list).reset_index()
procedure_codes_with_adm

,HADM_ID,ICD9_CODE
0,100003,"[4443, 9607, 9904, 3893]"
1,100006,"[9390, 9390, 9925, 9925]"
2,100007,"[4562, 5459]"
3,100009,"[3613, 3615, 3795, 3961]"
4,100010,"[5551, 540, 403]"
...,...,...
46836,199993,"[3614, 3512, 3761, 8842, 8848, 3961, 3964, 340..."
46837,199994,"[9671, 9604, 3995, 3891]"
46838,199995,"[3521, 3961, 3845, 3539, 8841, 8847, 9929, 887..."
46839,199998,"[3612, 3615, 3964]"


In [ ]:
# join notes with the procedure codes
notes_with_procedure_codes = pd.merge(notes_df, procedure_codes_with_adm, on='HADM_ID', how='inner')
notes_with_procedure_codes

,HADM_ID,CATEGORY,TEXT,ICD9_CODE
0,167853.0,Discharge summary,Admission Date: [**2151-7-16**] Dischar...,"[4542, 4542, 3491, 3491, 3893, 3893, 3891, 3891]"
1,107527.0,Discharge summary,Admission Date: [**2118-6-2**] Discharg...,"[9390, 966, 3199, 9671, 9604, 9605, 3323, 3324]"
2,167118.0,Discharge summary,Admission Date: [**2119-5-4**] D...,"[3179, 311, 9672, 9605, 9605, 9605, 9605, 14, ..."
3,196489.0,Discharge summary,Admission Date: [**2124-7-21**] ...,"[9672, 9604, 3891, 3893, 4513, 966]"
4,135453.0,Discharge summary,Admission Date: [**2162-3-3**] D...,"[8102, 8103, 353, 8163, 9671, 9604]"
...,...,...,...,...
53363,122526.0,Discharge summary,"Name: [**Known lastname 18311**], [**Known fi...","[3941, 3941, 3409, 3409, 9904, 9904, 8959, 8959]"
53364,135672.0,Discharge summary,"Name: [**Known lastname 18321**],[**Known fir...","[4562, 4562, 3893, 3893, 3893, 3893]"
53365,183951.0,Discharge summary,Name: [**Known lastname 18357**]-[**Known las...,"[3972, 3972, 221, 221, 125, 125, 212, 212, 884..."
53366,169165.0,Discharge summary,"Name: [**Known lastname **],[**Known firstnam...","[3521, 3521, 3611, 3611, 3799, 3799, 3961, 3961]"


In [20]:
# convert the notes and code array into a formatted jsonl
import json
with open(f"{DATA_PATH}/discharge_summary_notes_with_procedure_codes.jsonl", 'w') as f:
    for _, row in notes_with_procedure_codes.iterrows():
        entry = {
            "HADM_ID": row['HADM_ID'],
            "ICD9_CODES": row['ICD9_CODE'],
            "TEXT": row['TEXT']
        }
        f.write(json.dumps(entry) + '\n')
